# Q : 1
loading uningram and bigram probability distribution

In [15]:
import pickle

# Load unigram probabilities
with open("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/ngram_models/1gram_counts.pkl", "rb") as f:
    uni_counts = pickle.load(f)

# Load bigram probabilities
with open("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/ngram_models/2gram_counts.pkl", "rb") as f:
    bi_counts = pickle.load(f)


In [21]:
i=0
for key,value in uni_counts.items():
    print(key,":",value)
    if i>5:
        break
    i+=1
    

('આ',) : 15822
('વીડિયો',) : 418
('જુઓ:',) : 16
('ઊંઝા',) : 11
('માર્કેટયાર્ડ',) : 3
('આજથી',) : 124
('25',) : 174


# read validation and test sentences 

In [26]:
# Read sentences from a text file
def read_sentences(filename):
    sentences = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            tokens = line.strip().split()  # simple whitespace tokenization
            if tokens:
                sentences.append(tokens)
    return sentences

validation_sentences = read_sentences("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/validation_set_1lakh.txt")
test_sentences = read_sentences("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/test_set_1lakh.txt")


# bigrams for validation and test sentences 

In [27]:
def get_all_bigrams(sentences):
    all_bigrams = []
    for sent in sentences:
        for i in range(len(sent) - 1):
            all_bigrams.append((sent[i], sent[i+1]))
    return all_bigrams

val_bigrams = get_all_bigrams(validation_sentences)
test_bigrams = get_all_bigrams(test_sentences)


# compute pmi of seen bigrams

In [28]:
import math

def compute_pmi_with_smoothing(bigrams_list, uni_counts, bi_counts):
    pmi_scores = {}

    # Vocabulary size (number of unique words)
    V = len(uni_counts)
    
    # Total counts
    total_words = sum(uni_counts.values())
    total_bigrams = sum(bi_counts.values())

    for w1, w2 in bigrams_list:
        # Unigram probabilities with add-one smoothing
        p_w1 = (uni_counts.get((w1,), 0) + 1) / (total_words + V)
        p_w2 = (uni_counts.get((w2,), 0) + 1) / (total_words + V)

        # Bigram probabilities with add-one smoothing
        p_w1w2 = (bi_counts.get((w1, w2), 0) + 1) / (total_bigrams + V*V)

        # PMI formula
        pmi = math.log2(p_w1w2 / (p_w1 * p_w2))
        pmi_scores[(w1, w2)] = pmi

    return pmi_scores


# Compute PMI for validation and test sets
val_pmi = compute_pmi_with_smoothing(val_bigrams, uni_counts, bi_counts)
test_pmi = compute_pmi_with_smoothing(test_bigrams, uni_counts, bi_counts)


In [29]:
import pandas as pd

def save_pmi_scores(pmi_dict, filename):
    df = pd.DataFrame(pmi_dict.items(), columns=['bigram', 'PMI'])
    df.to_csv(filename, index=False)

save_pmi_scores(val_pmi, "validation_pmi.csv")
save_pmi_scores(test_pmi, "test_pmi.csv")


In [30]:
val = pd.read_csv("validation_pmi.csv")
val

,bigram,PMI
0,"('૭પ)', '(નિવૃત')",6.655735
1,"('(નિવૃત', 'જીઇબી')",5.655735
2,"('જીઇબી', 'એન્જીનીયર)')",5.655735
3,"('એન્જીનીયર)', 'તે')",-6.171806
4,"('તે', 'હીરેશનભાઇ,')",-6.171806
...,...,...
11309,"('મેન્યુફેક્ચરિંગ', 'થવાથી')",-2.772625
11310,"('થવાથી', 'આયાત')",-5.002923
11311,"('આયાત', 'ખર્ચ')",-7.335698
11312,"('ખર્ચ', 'બચી')",-7.471050


# Q : 2

# load and preprocess the data 

In [31]:
# Function to read sentences from a .txt file and tokenize
def read_and_tokenize(filename):
    sentences = []
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            # Strip leading/trailing whitespace and split by space
            tokens = line.strip().split()
            if tokens:  # skip empty lines
                sentences.append(tokens)
    return sentences

# Example usage
train_sentences = read_and_tokenize("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/train_set_1lakh.txt")
validation_sentences = read_and_tokenize("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/validation_set_1lakh.txt")
test_sentences = read_and_tokenize("C:/CLG LAB/5TH SEM/NLP/lab5/saved_data/test_set_1lakh.txt")

# Check first few sentences
print("Train:", train_sentences[:2])
print("Validation:", validation_sentences[:2])
print("Test:", test_sentences[:2])


Train: [['આ', 'વીડિયો', 'જુઓ:', 'ઊંઝા', 'માર્કેટયાર્ડ', 'આજથી', '25', 'જુલાઈ', 'સુધી', 'બંધ'], ['મિથેનોલ', 'આવ્યો', 'ક્યાંથી?']]
Validation: [['૭પ)', '(નિવૃત', 'જીઇબી', 'એન્જીનીયર)', 'તે', 'હીરેશનભાઇ,', 'ચેતનભાઇ', 'ના', 'પિતાશ્રીનું', 'તા.'], ['ભાવનગરનાં', 'નિલમબાગ', 'પોલીસ', 'મથકમાં', 'અધેવાડામાં', 'રહેતા', 'ઇન્દ્વજીતસિંહ', 'ઉર્ફે', 'ઇનો', 'વિક્રમસિંહ', 'ગોહિલ', 'કાચા', 'કામના', 'કેદી', 'તરીકે', 'ભાવનગર', 'જેલમાં', 'સજા', 'ભોગી', 'રહ્યો', 'હતો.']]
Test: [['પ્રણવ', 'મુખરજીએ', 'હોસ્પિટલની', 'તકતીનું', 'અનાવરણ', 'કર્યું', 'તહું.'], ['9', 'ટકા', 'સુધીનો', 'વધારો', 'નોંધાયો', 'હતો.']]


# build TF and IDF from training data

In [34]:
from collections import defaultdict
import math

# Build DF (document frequency) from training data
def compute_df(sentences):
    df = defaultdict(int)
    for sent in sentences:
        unique_tokens = set(sent)
        for token in unique_tokens:
            df[token] += 1
    return df

train_df = compute_df(train_sentences)
N_train = len(train_sentences)  # total number of training sentences

# Compute IDF from DF
idf = {}
for word, freq in train_df.items():
    idf[word] = math.log((N_train + 1) / (freq + 1)) + 1  # smoothed IDF


In [35]:
i=0
for word,freq in idf.items():
    print(word,":",freq)
    if i>3: 
        break
    i+=1

વીડિયો : 6.486379802080548
સુધી : 4.791532780824834
જુઓ: : 9.659519617626064
ઊંઝા : 10.00782631189428
25 : 7.345238484868827


# TF-IDF for sentences 

In [37]:
def compute_tf_idf(sentences, idf_dict):
    tfidf_vectors = []
    for sent in sentences:
        tf = defaultdict(int)
        for token in sent:
            tf[token] += 1
        # Normalize TF by sentence length
        tf_normalized = {word: count / len(sent) for word, count in tf.items()}
        # Compute TF-IDF
        tfidf = {word: tf_normalized[word] * idf_dict.get(word, math.log((N_train+1)/1)+1)
                 for word in tf_normalized}  # unknown words get minimal IDF
        tfidf_vectors.append(tfidf)
    return tfidf_vectors


In [38]:
train_tfidf = compute_tf_idf(train_sentences, idf)
validation_tfidf = compute_tf_idf(validation_sentences, idf)
test_tfidf = compute_tf_idf(test_sentences, idf)

# Example: print TF-IDF of first training sentence
print(train_tfidf[0])


{'આ': 0.2889135024705678, 'વીડિયો': 0.6486379802080549, 'જુઓ:': 0.9659519617626064, 'ઊંઝા': 1.0007826311894281, 'માર્કેટયાર્ડ': 1.110643860056239, 'આજથી': 0.7705241218900234, '25': 0.7345238484868828, 'જુલાઈ': 0.820227352053389, 'સુધી': 0.4791532780824834, 'બંધ': 0.5815649500435145}


# Convert TF-IDF dictionaries to aligned vectors

In [44]:
from scipy.sparse import lil_matrix, save_npz, load_npz

def tfidf_dicts_to_sparse(tfidf_list, vocab):
    """
    Converts a list of TF-IDF dictionaries to a sparse LIL matrix.
    """
    matrix = lil_matrix((len(tfidf_list), len(vocab)), dtype=float)
    word_index = {word: idx for idx, word in enumerate(vocab)}
    
    for i, tfidf in enumerate(tfidf_list):
        for word, value in tfidf.items():
            if word in word_index:
                matrix[i, word_index[word]] = value
    return matrix


In [ ]:
#convert to csr for fast

In [45]:
train_sparse = tfidf_dicts_to_sparse(train_tfidf, vocab).tocsr()
validation_sparse = tfidf_dicts_to_sparse(validation_tfidf, vocab).tocsr()
test_sparse = tfidf_dicts_to_sparse(test_tfidf, vocab).tocsr()


In [ ]:
# save it 

In [46]:
save_npz("train_tfidf.npz", train_sparse)
save_npz("validation_tfidf.npz", validation_sparse)
save_npz("test_tfidf.npz", test_sparse)

# To load later:
validation_sparse = load_npz("validation_tfidf.npz")
test_sparse = load_npz("test_tfidf.npz")


# store in harddisk

In [40]:
import pickle

# Save TF-IDF dictionaries
with open("train_tfidf.pkl", "wb") as f:
    pickle.dump(train_tfidf, f)

with open("validation_tfidf.pkl", "wb") as f:
    pickle.dump(validation_tfidf, f)

with open("test_tfidf.pkl", "wb") as f:
    pickle.dump(test_tfidf, f)

# To load later
# with open("train_tfidf.pkl", "rb") as f:
#     train_tfidf_loaded = pickle.load(f)


# Q : 3

In [49]:
from sklearn.neighbors import NearestNeighbors
import pandas as pd

def find_nearest_neighbors(tfidf_matrix, sentences, top_n_samples=3, save_path=None):
    """
    Finds nearest neighbor for each sentence and optionally saves results.
    """
    # Fit NearestNeighbors model
    nn_model = NearestNeighbors(n_neighbors=2, metric='cosine')
    nn_model.fit(tfidf_matrix)
    
    # Get nearest neighbors
    distances, indices = nn_model.kneighbors(tfidf_matrix)
    
    # Store original + nearest neighbor
    results = []
    for i, neighbors in enumerate(indices):
        nearest_idx = neighbors[1]  # skip the first neighbor (itself)
        results.append((sentences[i], sentences[nearest_idx]))
    
    # Print a few sample sentences
    print(f"\nSample {top_n_samples} nearest neighbors:")
    for s1, s2 in results[:top_n_samples]:
        print("Original: ", s1)
        print("Nearest Neighbor: ", s2)
        print()
    
    # Save to disk
    if save_path:
        df = pd.DataFrame(results, columns=["Original", "NearestNeighbor"])
        df.to_csv(save_path, index=False)
    
    return results

# --- Apply to validation set ---
validation_results = find_nearest_neighbors(
    validation_sparse, validation_sentences,
    top_n_samples=3,
    save_path="validation_nearest_neighbors.csv"
)

# --- Apply to test set ---
test_results = find_nearest_neighbors(
    test_sparse, test_sentences,
    top_n_samples=3,
    save_path="test_nearest_neighbors.csv"
)



Sample 3 nearest neighbors:
Original:  ['૭પ)', '(નિવૃત', 'જીઇબી', 'એન્જીનીયર)', 'તે', 'હીરેશનભાઇ,', 'ચેતનભાઇ', 'ના', 'પિતાશ્રીનું', 'તા.']
Nearest Neighbor:  ['તા.']

Original:  ['ભાવનગરનાં', 'નિલમબાગ', 'પોલીસ', 'મથકમાં', 'અધેવાડામાં', 'રહેતા', 'ઇન્દ્વજીતસિંહ', 'ઉર્ફે', 'ઇનો', 'વિક્રમસિંહ', 'ગોહિલ', 'કાચા', 'કામના', 'કેદી', 'તરીકે', 'ભાવનગર', 'જેલમાં', 'સજા', 'ભોગી', 'રહ્યો', 'હતો.']
Nearest Neighbor:  ['પોલીસ', 'તંત્ર', 'ડિસિપ્લિન', 'ફોર્સ', 'તરીકે', 'છે.']

Original:  ['ઓટોરિક્ષા', 'ચાલકોને', 'નવી', 'ઓળખ', 'મળી', 'ગઈ', 'છે.']
Nearest Neighbor:  ['નવી', 'દિલ્હી,', 'તા.']


Sample 3 nearest neighbors:
Original:  ['પ્રણવ', 'મુખરજીએ', 'હોસ્પિટલની', 'તકતીનું', 'અનાવરણ', 'કર્યું', 'તહું.']
Nearest Neighbor:  ['આ', 'ફિલ્મથી', 'સારાએ', 'બોલિવુડ', 'ડેબ્યૂ', 'કર્યું', 'હતું.']

Original:  ['9', 'ટકા', 'સુધીનો', 'વધારો', 'નોંધાયો', 'હતો.']
Nearest Neighbor:  ['ગાંધીધામમાં', 'લાંબા', 'સમય', 'બાદ', 'કોરોનાનો', 'એકપણ', 'કેસ', 'નોંધાયો', 'ન', 'હતો.']

Original:  ['.?']
Nearest Neighbor:  ['વિટામિન